In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ptypy.utils.metrics import fsc, frc_threshold, compute_intersection
from scipy.ndimage import gaussian_filter

#### Generate the volumes

Here we generate:
- a first volume with random numbers
- a second volume by adding gaussian noise to the first

In [ ]:
# Add gaussian noise to the image
def add_gaussian_noise(volume, mean=0.0, std=0.1):
    noise = np.random.normal(mean, std, volume.shape)
    return volume + noise

original_vol = gaussian_filter(np.random.randn(110,120,100), sigma=3)
noisy_vol = add_gaussian_noise(original_vol, mean=0.0, std=0.03)

#### Visualise the volumes

In [ ]:
def plot_vol(noisy_vol, original, title=''):
    
    pshape = noisy_vol.shape[0]

    pos_limit = np.max(original) 
    neg_limit = np.min(original)
    
    fig, axes = plt.subplots(ncols=3, nrows=2, figsize=(6,4), dpi=100)
    for i in range(3):
        for j in range(2):
            ax = axes[j,i]
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xticklabels([])
            ax.set_yticklabels([])
    axes[0,0].set_title("slice(Z)")
    axes[0,1].set_title("slice(Y)")
    axes[0,2].set_title("slice(X)")
    axes[0,0].set_ylabel("Original vol")
    axes[0,0].imshow(original[pshape//2], vmin=neg_limit, vmax=pos_limit)
    axes[0,1].imshow(original[:,pshape//2], vmin=neg_limit, vmax=pos_limit)
    axes[0,2].imshow(original[:,:,pshape//2], vmin=neg_limit, vmax=pos_limit)
    axes[1,0].set_ylabel("Noisy vol")
    axes[1,0].imshow(noisy_vol[pshape//2], vmin=neg_limit, vmax=pos_limit)
    axes[1,1].imshow(noisy_vol[:,pshape//2], vmin=neg_limit, vmax=pos_limit)
    im1 = axes[1,2].imshow(noisy_vol[:,:,pshape//2], vmin=neg_limit, vmax=pos_limit)
    fig.suptitle('Volumes')
    fig.colorbar(im1, ax=axes.ravel().tolist())


In [ ]:
plot_vol(noisy_vol, original_vol)

#### Compute the FSC and plot

The amount of noise added to generate volume 2 was relatively small, and this is reflected in the FSC curve.

In [ ]:
X, FSC, N = fsc(
    noisy_vol, 
    original_vol, 
    apod_width = 0, 
    ringthick=2, 
)
threshold = frc_threshold(N)
resolution = compute_intersection(X, FSC, threshold)

In [ ]:
plt.figure()
plt.plot(X, FSC, color="b", label="FSC")
plt.plot(X, threshold, color="r", ls="--", label="1 bit threshold")
plt.xlim(0,0.5)
plt.legend()
if resolution is not None:
    plt.axvline(resolution, color="k", ls=":")
plt.show()

#### Increasing noise in the second volume

Now we generate another volume, still by adding noise to the first volume, but a larger amount of noise. This change has a clear impact on the FSC curve.

In [ ]:
noisy_vol2 = add_gaussian_noise(original_vol, mean=0.0, std=0.07)
plot_vol(noisy_vol2, original_vol)

In [ ]:
X, FSC, N = fsc(
    noisy_vol2, 
    original_vol, 
    apod_width = 0, 
    ringthick=2, 
)
threshold = frc_threshold(N)
resolution = compute_intersection(X, FSC, threshold)

In [ ]:
plt.figure()
plt.plot(X, FSC, color="b", label="FSC")
plt.plot(X, threshold, color="r", ls="--", label="1 bit threshold")
plt.xlim(0,0.5)
plt.legend()
if resolution is not None:
    plt.axvline(resolution, color="k", ls=":")
plt.show()